# Seeing Predictor — Exploratory Data Analysis

Inspect the trained XGBoost atmospheric-seeing model: which weather features
drive the prediction, and how well it recovers the synthetic targets it was
trained on.

The model is currently trained on **synthetic bootstrap data** (see
`api/ml/train_xgb.py`). The final cell documents the upgrade path to real
ERA5 reanalysis once CDS credentials are configured.

## 1. Setup

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb

# Put the repo root on sys.path so `api.ml.*` imports resolve when the
# notebook runs from api/ml/notebooks/.
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from api.ml.features import FEATURE_NAMES
from api.ml.train_xgb import generate_synthetic_dataset

MODEL_PATH = os.path.join(REPO_ROOT, "api", "ml", "models", "seeing_model.json")
print("repo root:", REPO_ROOT)
print("model exists:", os.path.exists(MODEL_PATH))

## 2. Load the trained model

In [ ]:
booster = xgb.Booster()
booster.load_model(MODEL_PATH)
print("num features:", booster.num_features())
print("feature names:", booster.feature_names)

## 3. Feature importance

Which of the 24 features the trees actually split on. Expect the wind,
temperature-variability, cloud-fraction and dewpoint-depression features to
dominate — those are the LAMOST-paper drivers the synthetic targets are
generated from.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
xgb.plot_importance(booster, ax=ax, importance_type="gain", show_values=False)
ax.set_title("Feature importance (gain)")
plt.tight_layout()
plt.show()

## 4. Synthetic data validation

Regenerate a fresh batch of synthetic samples (different seed than training)
and compare predicted vs. actual seeing.

In [ ]:
X, y = generate_synthetic_dataset(n_samples=1000, seed=123)
dmatrix = xgb.DMatrix(X, feature_names=list(FEATURE_NAMES))
preds = booster.predict(dmatrix)

err = preds - y
mae = float(np.mean(np.abs(err)))
rmse = float(np.sqrt(np.mean(err ** 2)))
ss_res = float(np.sum(err ** 2))
ss_tot = float(np.sum((y - np.mean(y)) ** 2))
r2 = 1.0 - ss_res / ss_tot
print(f"MAE  : {mae:.4f} arcsec")
print(f"RMSE : {rmse:.4f} arcsec")
print(f"R^2  : {r2:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y, preds, s=8, alpha=0.3)
lims = [min(y.min(), preds.min()), max(y.max(), preds.max())]
ax.plot(lims, lims, "r--", linewidth=1, label="perfect")
ax.set_xlabel("Actual seeing (arcsec)")
ax.set_ylabel("Predicted seeing (arcsec)")
ax.set_title("Predicted vs. actual seeing (synthetic val)")
ax.legend()
plt.tight_layout()
plt.show()

## 5. TODO — upgrade to ERA5 reanalysis

The synthetic generator is a bootstrap. To train on real atmospheric data,
pull [ERA5 reanalysis](https://cds.climate.copernicus.eu/) via the CDS API and
replace `generate_synthetic_dataset()` with a loader that:

1. Downloads hourly single-level variables over a grid of known observatory
   sites (2m temperature, 2m dewpoint, MSL pressure, 10m u/v wind, total
   cloud cover).
2. Joins each weather window to a **measured** seeing value (e.g. DIMM
   monitor archives, or the LAMOST seeing logs) as the regression target.
3. Feeds each window through `build_feature_vector()` exactly as the
   synthetic path does, so the 24-feature contract is unchanged.

### CDS API call template

```python
# pip install cdsapi  (and put your key in ~/.cdsapirc)
import cdsapi

c = cdsapi.Client()
c.retrieve(
    "reanalysis-era5-single-levels",
    {
        "product_type": "reanalysis",
        "format": "netcdf",
        "variable": [
            "2m_temperature",
            "2m_dewpoint_temperature",
            "mean_sea_level_pressure",
            "10m_u_component_of_wind",
            "10m_v_component_of_wind",
            "total_cloud_cover",
        ],
        "year": ["2022", "2023"],
        "month": [f"{m:02d}" for m in range(1, 13)],
        "day": [f"{d:02d}" for d in range(1, 32)],
        "time": [f"{h:02d}:00" for h in range(24)],
        "area": [lat + 0.25, lon - 0.25, lat - 0.25, lon + 0.25],  # N, W, S, E
    },
    "era5_site.nc",
)
```

Then re-run `python -m api.ml.train_xgb` (pointing it at the ERA5 loader)
and re-evaluate here.